In [3]:
from pathlib import Path
import csv
from datetime import date
import sqlite3

import shutil

In [4]:
master_db_path = Path("../databases/Master_Database.db").resolve()
master_conn = sqlite3.connect(master_db_path)

stackX3_source = Path("../py_programs/stackX3/stackX3.py")

export_folder = Path("../lab batches/sx3_03")


#Where "None" means no filter
target_objects = None #["SN2019tgm", "SN2026atb", "SN2020SE"]
target_groups = None          

n_lanes = 6

In [5]:
def get_mosaic_work(master_conn, target_objects=None, target_groups=None):
    sql = """
        SELECT group_id, object, obs_date, telescope,
                filter, img_start, img_end, num_exps
        FROM observations
        WHERE mosaic_name_f IS NULL
            AND error_f IS NULL
        
    """

    params = []

    if target_objects is not None:
        marks = ", ".join("?" * len(target_objects))
        
        sql += f" AND object IN ({marks})"
        params += [name.upper() for name in target_objects]

    if target_groups is not None:
        marks = ", ".join("?" * len(target_groups))
        sql += f" AND group_id IN ({marks})"
        params += list(target_groups)

    
    sql += " ORDER BY group_id"



    
    work_list = master_conn.execute(sql, params).fetchall()

    return work_list

In [6]:
flat_overrides = {
    # ("west-14", "B", "2026"): "none",
}


def get_override(telescope, band, obs_date):
    return flat_overrides.get((telescope, band, obs_date[:4]))

In [7]:
def get_exported_groups(batches_folder, export_folder):
    
    exported = {}

    
    for manifest_path in sorted(batches_folder.glob("*/manifest.csv")):

        
        if manifest_path.parent.resolve() == export_folder.resolve():
            continue

        
        with manifest_path.open(newline="", encoding="utf-8") as file:
            for record in csv.DictReader(file):
                exported[int(record["group_id"])] = manifest_path.parent.name

    return exported


def assign_lanes(ready_list, n_lanes):
    
    frames_by_object = {}
    for row in ready_list:
        object_name, num_exps = row[1], row[7]
        frames_by_object[object_name] = frames_by_object.get(object_name, 0) + num_exps

    
    lane_frames = [0] * n_lanes
   
    lane_of_object = {}

    
    for object_name in sorted(frames_by_object, key=frames_by_object.get, reverse=True):
        lane = lane_frames.index(min(lane_frames))   
        lane_of_object[object_name] = lane
        lane_frames[lane] += frames_by_object[object_name]

    
    lanes = [[] for _ in range(n_lanes)]
    for row in ready_list:
        lanes[lane_of_object[row[1]]].append(row)

   
    return lanes, lane_frames

In [8]:
def make_mosaic_commands(group_id, telescope, lab_date,
                         img_start, img_end, filename, override):

    
    if override is None:
        flat_arg = ""
    else:
        flat_arg = f' "{override}"'

    
    noastrom_name = filename.replace("-mosaic.fits", "-noastrom.fits")

    
    return f'''
echo "Processing group {group_id}"

if ( -e "$batch_dir/output_mosaics/{filename}" ) then
   echo "Output already exists; not overwriting group {group_id}"

else if ( -d "$batch_dir/working/g{group_id}" ) then
   echo "Work folder already exists; inspect logs/g{group_id}.log"

else
   mkdir "$batch_dir/working/g{group_id}"
   if ( $status != 0 ) exit 1

   cd "$batch_dir/working/g{group_id}"
   if ( $status != 0 ) exit 1

   python "$batch_dir/stackX3.py" "{telescope}" "{lab_date}" "{img_start}" "{img_end}" 5.0{flat_arg} >&! "$batch_dir/logs/g{group_id}.log"
   set stack_status = $status
   echo "stackX3 exit status: $stack_status" >> "$batch_dir/logs/g{group_id}.log"

   grep -q N_ASTARS amosaic.fits >& /dev/null
   set astrom_status = $status

   if ( $stack_status == 0 && $astrom_status == 0 ) then
       mv amosaic.fits "$batch_dir/output_mosaics/{filename}"
       if ( $status != 0 ) exit 1

       echo "Saved group {group_id}"

   else if ( -e mosaic.fits ) then
       cp mosaic.fits "$batch_dir/stacked_no_astrom/{noastrom_name}"
       echo "Stacked but no astrometry for group {group_id}; see logs/g{group_id}.log"

   else
       echo "Failed group {group_id}; see logs/g{group_id}.log"
   endif

   cd "$batch_dir"
   if ( $status != 0 ) exit 1
endif
'''

In [9]:
def write_batch(lanes, export_folder, stackX3_source):

    
    if not stackX3_source.is_file():
        raise FileNotFoundError(f"stackX3 not found at {stackX3_source}")

    
    if export_folder.exists():
        raise FileExistsError(f"{export_folder} already exists; pick a new export_folder")

    
    export_folder.mkdir(parents=True)

    
    stackx3_copy = export_folder / "stackX3.py"
    shutil.copy2(stackX3_source, stackx3_copy)

    
    

    today = date.today().isoformat()   
    manifest_rows = []                 
    script_paths = []                  

    
    for lane_number, lane_rows in enumerate(lanes, start=1):

        
        if not lane_rows:
            continue

        script_name = f"run_lane{lane_number}.csh"




        
        output = [
            "#!/bin/csh -f",
            'set batch_dir = "$cwd"',
            'if ( ! -f "$batch_dir/stackX3.py" ) then',
            '   echo "Run this from inside the batch folder; stackX3.py not found"',
            "   exit 1",
            "endif",
            "mkdir -p working output_mosaics logs stacked_no_astrom",
            "if ( $status != 0 ) exit 1",
        ]

        for row in lane_rows:
            
            (group_id, object_name, obs_date, telescope, band,
             img_start, img_end, num_exps, override) = row

            
            lab_date = date.fromisoformat(obs_date).strftime("%y_%m_%d")
        
            target = object_name.lower()


            
            filename = (
                f"{target}-{lab_date}-{telescope}-{band}"
                f"-flat-g{group_id}-mosaic.fits"
            )

            
            output.append(make_mosaic_commands(
                group_id, telescope, lab_date,
                img_start, img_end, filename, override,
            ))

            
            manifest_rows.append([
                group_id, object_name, lane_number, script_name,
                obs_date, telescope, band, img_start, img_end, num_exps,
                override or "", filename, today,
            ])

      
        output.append(f'echo "Lane {lane_number} finished"')

       
        script_path = export_folder / script_name
        script_path.write_text("\n".join(output) + "\n", encoding="utf-8")
       
        script_path.chmod(0o755)
        script_paths.append(script_path)


    manifest_path = export_folder / "manifest.csv"
    with manifest_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow([
            "group_id", "object", "lane", "script",
            "obs_date", "telescope", "filter", "img_start", "img_end", "num_exps",
            "flat_override", "filename", "exported",
        ])
        writer.writerows(manifest_rows)

    return script_paths, manifest_path

In [10]:
allow_reexport = False

work_list = get_mosaic_work(master_conn, target_objects, target_groups)

if target_objects is not None:
    found = {row[1] for row in work_list}          
    missing = [name for name in target_objects if name.upper() not in found]
    if missing:
        print("No pending groups for:", missing)


ready_list = []
for row in work_list:
    group_id, object_name, obs_date, telescope, band, img_start, img_end, num_exps = row
    override = get_override(telescope, band, obs_date)
    ready_list.append(row + (override,))   


exported = get_exported_groups(export_folder.parent, export_folder)
already = [row for row in ready_list if row[0] in exported]

if already:
    print("Already in an earlier export but not imported yet:", len(already))
    for row in already[:20]:                       
        print("  ", row[0], row[1], "in", exported[row[0]])
    if not allow_reexport:
       
        raise RuntimeError("Set allow_reexport = True to export these groups again")


lanes, lane_frames = assign_lanes(ready_list, n_lanes)


print("Groups to export:", len(ready_list))
print("With a flat override:", sum(1 for row in ready_list if row[8] is not None))


for lane_number, (lane_rows, frames) in enumerate(zip(lanes, lane_frames), start=1):
    counts = {}
    for row in lane_rows:
        counts[row[1]] = counts.get(row[1], 0) + 1

    print(f"Lane {lane_number}: {len(lane_rows)} groups, {frames} frames")
    for object_name, count in sorted(counts.items()):
        print("    ", object_name, count)


big_groups = [row for row in ready_list if row[7] >= 100]
for row in big_groups:
    print("Large group:", row[0], row[1], row[7], "frames")

Groups to export: 1845
With a flat override: 0
Lane 1: 349 groups, 1772 frames
     AT2026AHAB 2
     SN2019AIK 264
     SN2026KC 83
Lane 2: 217 groups, 1838 frames
     SN2018GV 90
     SN2019NP 34
     SN2024XYN 93
Lane 3: 338 groups, 1772 frames
     SN2017HHZ 70
     SN2018KSS 168
     SN2025AHSA 98
     SN2026AFSY 1
     SN2026AICD 1
Lane 4: 329 groups, 1768 frames
     AT2026AHVI 1
     SN2020UEC 144
     SN2020VEY 83
     SN2025AHKT 99
     SN2026AEIQ 1
     SN2026AFRP 1
Lane 5: 352 groups, 1911 frames
     SN2017MS 92
     SN2020WCJ 68
     SN2024YM 91
     SN2026GM 101
Lane 6: 260 groups, 1768 frames
     AT2026AFSY 1
     AT2026AGUA 2
     SN2017HN 66
     SN2021XN 70
     SN2024ANA 116
     SN2026AEPK 1
     SN2026AESJ 2
     SN2026AFNJ 2
Large group: 2021 SN2024XYN 188 frames
Large group: 2027 SN2024XYN 148 frames
Large group: 2064 SN2024XYN 185 frames


In [11]:
script_paths, manifest_path = write_batch(lanes, export_folder, stackX3_source)

print("Batch folder:", export_folder.resolve())
for path in script_paths:
    print("  ", path.name)
print("Manifest:", manifest_path.name)

Batch folder: /Users/mehtabsingh/Desktop/Desktop/Durham/3rd Year/Modules/Astro Lab/Astrolab_Code/lab batches/sx3_03
   run_lane1.csh
   run_lane2.csh
   run_lane3.csh
   run_lane4.csh
   run_lane5.csh
   run_lane6.csh
Manifest: manifest.csv
